LTDL assimilation and rasterization using taget raster

lets make sure that the interpreter and the kernel are working (make sure to select the interpreter first using comm + shift + P and use the arcproclone path)

In [ ]:
#evaluate if the arcpro interpreter and kernel are live
import sys
import arcpy

print(sys.executable)
print(arcpy.GetInstallInfo()["Version"])
print(arcpy.env.workspace)

c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcproclone\python.exe
3.5
None


In [3]:
import arcpy
import os

arcpy.env.workspace = r"C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_assimilation\LTDL_assimilation.gdb"
arcpy.env.overwriteOutput = True

print("Workspace:", arcpy.env.workspace)
print("Scratch GDB:", arcpy.env.scratchGDB)
print("Scratch Folder:", arcpy.env.scratchFolder)

Workspace: C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_assimilation\LTDL_assimilation.gdb
Scratch GDB: C:\Users\SCOTTF~1\AppData\Local\Temp\scratch.gdb
Scratch Folder: C:\Users\SCOTTF~1\AppData\Local\Temp\scratch


In [4]:
import arcpy, os

gdb = r"C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_Release_Sept2024.gdb"
arcpy.env.workspace = gdb

print(arcpy.ListFeatureClasses())
print(arcpy.ListTables())

['LTDL_Project_Polygons', 'LTDL_Project_Lines', 'LTDL_Project_Points', 'LTDL_Treatment_Lines', 'LTDL_Treatment_Points', 'LTDL_Treatment_Polygons']
['project_info', 'treatment_info', 'equipment_used', 'herbicide', 'project_identifiers', 'related_treatments', 'seed_species_vendor_info', 'seed_species', 'related_project']


In [5]:
import arcpy, os

input_fc = r"C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_Release_Sept2024.gdb\ltdl_treatment_polygons"
out_fc = r"memory\test_proj"
target = arcpy.Describe(r"C:\NCA_DATA\templates\grid_template_10m_26911_uint8.tif").spatialReference

if arcpy.Exists(out_fc):
    arcpy.management.Delete(out_fc)

arcpy.management.Project(input_fc, out_fc, target)

print(arcpy.management.GetCount(out_fc))
print(arcpy.Describe(out_fc).spatialReference.name)

844
NAD_1983_UTM_Zone_11N


In [9]:
#write a geodatabase for the vector datasets
import arcpy
import os

vector_folder = r"C:\NCA_DATA\Ancillary_Data\LTDL"
vector_gdb_name = "LTDL_vspy.gdb"
vector_gdb = os.path.join(vector_folder, vector_gdb_name)

if not arcpy.Exists(vector_gdb):
    arcpy.management.CreateFileGDB(vector_folder, vector_gdb_name)

print("Vector GDB:", vector_gdb)

Vector GDB: C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_vspy.gdb


In [11]:
import arcpy
import os
import re
import gc
from collections import OrderedDict
from arcpy.sa import Raster, CellStatistics, Con, IsNull, SetNull

arcpy.CheckOutExtension("Spatial")

# =============================================================================
# USER INPUTS
# =============================================================================

input_gdb = r"C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_Release_Sept2024.gdb"
base_folder = r"C:\NCA_DATA\Ancillary_Data\LTDL"
vector_gdb_name = "LTDL_vspy.gdb"

aoi_fc = r"C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_assimilation\LTDL_assimilation.gdb\NCA_NAD83UTM11N"
template_raster = r"C:\NCA_DATA\templates\grid_template_10m_26911_uint8.tif"

treatment_polygons = os.path.join(input_gdb, "ltdl_treatment_polygons")
treatment_info = os.path.join(input_gdb, "treatment_info")

analysis_year = 2026

# =============================================================================
# CREATE / DEFINE OUTPUT LOCATIONS
# =============================================================================

vector_gdb = os.path.join(base_folder, vector_gdb_name)
if not arcpy.Exists(vector_gdb):
    arcpy.management.CreateFileGDB(base_folder, vector_gdb_name)

raster_folder = base_folder

clip_fc_new = os.path.join(vector_gdb, "ltdl_treatments_clip_new")

out_rasters = {
    "MGMT_BIN":     os.path.join(raster_folder, "mgmt_binary_10m_new.tif"),
    "TIME_SINCE":   os.path.join(raster_folder, "mgmt_time_since_10m_new.tif"),
    "SEEDED_BIN":   os.path.join(raster_folder, "mgmt_seeded_10m_new.tif"),
    "NUM_UNITS_R":  os.path.join(raster_folder, "mgmt_num_units_10m_new.tif"),
    "TRT_MAJOR_ID": os.path.join(raster_folder, "mgmt_trt_major_10m_new.tif"),
    "TRT_SUB_ID":   os.path.join(raster_folder, "mgmt_trt_sub_10m_new.tif"),
    "TRT_TYPE_ID":  os.path.join(raster_folder, "mgmt_treatment_type_10m_new.tif"),
}

mgmt_most_recent_year_path = os.path.join(raster_folder, "mgmt_most_recent_year_10m_new.tif")
mgmt_most_recent_time_since_path = os.path.join(raster_folder, "mgmt_most_recent_time_since_10m_new.tif")
mgmt_times_treated_path = os.path.join(raster_folder, "mgmt_times_treated_10m_new.tif")

lookup_tables = {
    "Trt_Type_Major": os.path.join(vector_gdb, "lut_Trt_Type_Major_new"),
    "Trt_Type_Sub":   os.path.join(vector_gdb, "lut_Trt_Type_Sub_new"),
    "Treatment_Type": os.path.join(vector_gdb, "lut_Treatment_Type_new"),
}

# =============================================================================
# ENVIRONMENT
# =============================================================================

arcpy.env.overwriteOutput = True
arcpy.env.workspace = vector_gdb
arcpy.env.snapRaster = template_raster
arcpy.env.extent = template_raster
arcpy.env.cellSize = template_raster
arcpy.env.outputCoordinateSystem = arcpy.Describe(template_raster).spatialReference

# =============================================================================
# HELPERS
# =============================================================================

def log(msg):
    print(msg)

def clear_locks():
    try:
        arcpy.management.ClearWorkspaceCache()
    except Exception:
        pass
    gc.collect()

def field_exists(table, name):
    return name in [f.name for f in arcpy.ListFields(table)]

def add_field_if_missing(table, name, ftype, **kwargs):
    if not field_exists(table, name):
        arcpy.management.AddField(table, name, ftype, **kwargs)

def assert_not_exists(path):
    if arcpy.Exists(path):
        raise RuntimeError(f"Output already exists. Delete or rename it first:\n{path}")

def polygon_to_raster(in_fc, value_field, out_path, priority_field="NONE", cell_assignment="CELL_CENTER"):
    assert_not_exists(out_path)
    arcpy.conversion.PolygonToRaster(
        in_features=in_fc,
        value_field=value_field,
        out_rasterdataset=out_path,
        cell_assignment=cell_assignment,
        priority_field=priority_field,
        cellsize=template_raster
    )
    log(f"Raster created: {out_path}")

def build_lookup(table, source_field, code_field, out_table):
    values = []
    with arcpy.da.SearchCursor(table, [source_field]) as cur:
        for (v,) in cur:
            if v is None:
                continue
            v = str(v).strip()
            if v == "":
                continue
            values.append(v)

    unique_vals = list(OrderedDict.fromkeys(sorted(set(values))))
    code_map = {val: i + 1 for i, val in enumerate(unique_vals)}

    add_field_if_missing(table, code_field, "LONG")

    with arcpy.da.UpdateCursor(table, [source_field, code_field]) as cur:
        for row in cur:
            src = row[0]
            if src is None or str(src).strip() == "":
                row[1] = 0
            else:
                row[1] = code_map[str(src).strip()]
            cur.updateRow(row)

    assert_not_exists(out_table)
    out_workspace = os.path.dirname(out_table)
    out_name = os.path.basename(out_table)

    arcpy.management.CreateTable(out_workspace, out_name)
    arcpy.management.AddField(out_table, code_field, "LONG")
    arcpy.management.AddField(out_table, source_field[:60], "TEXT", field_length=255)

    with arcpy.da.InsertCursor(out_table, [code_field, source_field[:60]]) as icur:
        for val, code in code_map.items():
            icur.insertRow([code, val])

    log(f"Lookup table created: {out_table}")

# =============================================================================
# VALIDATION
# =============================================================================

required_paths = [
    input_gdb,
    base_folder,
    vector_gdb,
    aoi_fc,
    template_raster,
    treatment_polygons,
    treatment_info,
]
for p in required_paths:
    if not arcpy.Exists(p):
        raise RuntimeError(f"Required dataset not found: {p}")

assert_not_exists(clip_fc_new)
for p in out_rasters.values():
    assert_not_exists(p)
assert_not_exists(mgmt_most_recent_year_path)
assert_not_exists(mgmt_most_recent_time_since_path)
assert_not_exists(mgmt_times_treated_path)
for p in lookup_tables.values():
    assert_not_exists(p)

# =============================================================================
# CLEAN SESSION LAYERS
# =============================================================================

for lyr_name in ["treatment_lyr", "yr_lyr"]:
    if arcpy.Exists(lyr_name):
        try:
            arcpy.management.Delete(lyr_name)
        except Exception:
            pass

clear_locks()

# =============================================================================
# STEP 1 — BUILD CLIPPED FEATURE CLASS IN MEMORY, THEN WRITE FINAL _NEW
# =============================================================================

target_sr = arcpy.Describe(template_raster).spatialReference

mem_proj = r"memory\ltdl_poly_26911"
mem_join = r"memory\ltdl_treatments_joined"
mem_clip = r"memory\ltdl_treatments_clip"
lyr = "treatment_lyr"

for p in [mem_proj, mem_join, mem_clip]:
    if arcpy.Exists(p):
        arcpy.management.Delete(p)

if arcpy.Exists(lyr):
    arcpy.management.Delete(lyr)

log("Projecting treatment polygons to template CRS in memory...")
arcpy.management.Project(
    treatment_polygons,
    mem_proj,
    target_sr
)

log("Copying projected polygons in memory...")
arcpy.management.CopyFeatures(mem_proj, mem_join)

candidate_join_fields = [
    "Prj_ID",
    "Trt_ID",
    "Plan_Imp",
    "Dates_Confirmed",
    "Init_Date",
    "Comp_Date",
    "Units",
    "Num_Units",
    "Trt_Type_Major",
    "Trt_Type_Sub",
    "Treatment_Type",
    "Treatment_Seeded",
    "Trt_Feature_Type",
]
join_fields = [f for f in candidate_join_fields if field_exists(treatment_info, f) and f != "Trt_ID"]

log(f"Joining treatment_info fields: {join_fields}")
arcpy.management.JoinField(
    mem_join,
    "Trt_ID",
    treatment_info,
    "Trt_ID",
    join_fields
)

arcpy.management.MakeFeatureLayer(mem_join, lyr)
arcpy.management.SelectLayerByAttribute(
    lyr,
    "NEW_SELECTION",
    "Plan_Imp = 'Implemented'"
)

log("Clipping implemented treatments to AOI in memory...")
arcpy.analysis.Clip(lyr, aoi_fc, mem_clip)

if arcpy.Exists(lyr):
    arcpy.management.Delete(lyr)
clear_locks()

arcpy.management.CopyFeatures(mem_clip, clip_fc_new)

for p in [mem_proj, mem_join, mem_clip]:
    if arcpy.Exists(p):
        arcpy.management.Delete(p)

clear_locks()

n = int(arcpy.management.GetCount(clip_fc_new)[0])
log(f"Features after clip: {n}")
if n == 0:
    raise RuntimeError("No implemented treatment polygons intersect the AOI.")

# =============================================================================
# STEP 2 — DERIVED FIELDS
# =============================================================================

add_field_if_missing(clip_fc_new, "MGMT_BIN", "SHORT")
arcpy.management.CalculateField(clip_fc_new, "MGMT_BIN", "1", "PYTHON3")

if field_exists(clip_fc_new, "Treatment_Seeded"):
    add_field_if_missing(clip_fc_new, "SEEDED_BIN", "SHORT")
    arcpy.management.CalculateField(
        clip_fc_new,
        "SEEDED_BIN",
        "1 if !Treatment_Seeded! == -1 else 0",
        "PYTHON3"
    )

add_field_if_missing(clip_fc_new, "TRT_YEAR", "SHORT")
year_code = r"""
import re
def extract_year(v):
    if not v:
        return None
    m = re.search(r'\d{4}', str(v))
    return int(m.group()) if m else None

def year_from_date(comp_date, init_date):
    return extract_year(comp_date) or extract_year(init_date)
"""
arcpy.management.CalculateField(
    clip_fc_new,
    "TRT_YEAR",
    "year_from_date(!Comp_Date!, !Init_Date!)",
    "PYTHON3",
    year_code
)

add_field_if_missing(clip_fc_new, "TIME_SINCE", "SHORT")
arcpy.management.CalculateField(
    clip_fc_new,
    "TIME_SINCE",
    f"{analysis_year} - !TRT_YEAR! if !TRT_YEAR! not in [None, ''] else None",
    "PYTHON3"
)

if field_exists(clip_fc_new, "Num_Units"):
    add_field_if_missing(clip_fc_new, "NUM_UNITS_R", "DOUBLE")
    arcpy.management.CalculateField(
        clip_fc_new,
        "NUM_UNITS_R",
        "!Num_Units!",
        "PYTHON3"
    )

# =============================================================================
# STEP 3 — ENCODE CATEGORICAL FIELDS + LOOKUPS
# =============================================================================

if field_exists(clip_fc_new, "Trt_Type_Major"):
    build_lookup(clip_fc_new, "Trt_Type_Major", "TRT_MAJOR_ID", lookup_tables["Trt_Type_Major"])

if field_exists(clip_fc_new, "Trt_Type_Sub"):
    build_lookup(clip_fc_new, "Trt_Type_Sub", "TRT_SUB_ID", lookup_tables["Trt_Type_Sub"])

if field_exists(clip_fc_new, "Treatment_Type"):
    build_lookup(clip_fc_new, "Treatment_Type", "TRT_TYPE_ID", lookup_tables["Treatment_Type"])

# =============================================================================
# STEP 4 — STANDARD RASTERS
# =============================================================================

log("Rasterizing standard outputs...")
for field_name, out_path in out_rasters.items():
    if field_exists(clip_fc_new, field_name):
        polygon_to_raster(clip_fc_new, field_name, out_path, priority_field="NONE")

# =============================================================================
# STEP 5 — MOST RECENT YEAR / TIME SINCE
# =============================================================================

years = sorted({
    row[0] for row in arcpy.da.SearchCursor(clip_fc_new, ["TRT_YEAR"])
    if row[0] is not None
})

if years:
    log("Rasterizing most recent treatment year...")
    polygon_to_raster(
        clip_fc_new,
        "TRT_YEAR",
        mgmt_most_recent_year_path,
        priority_field="TRT_YEAR"
    )

    mgmt_most_recent_year = Raster(mgmt_most_recent_year_path)

    log("Deriving most recent time since raster...")
    mgmt_most_recent_time_since = SetNull(
        IsNull(mgmt_most_recent_year),
        analysis_year - mgmt_most_recent_year
    )
    mgmt_most_recent_time_since.save(mgmt_most_recent_time_since_path)

# =============================================================================
# STEP 6 — TIMES TREATED (YEAR-GROUP APPROXIMATION; MEMORY ONLY TEMPS)
# =============================================================================

if years:
    log("Building times treated raster...")
    yr_lyr = "yr_lyr"
    if arcpy.Exists(yr_lyr):
        arcpy.management.Delete(yr_lyr)
    arcpy.management.MakeFeatureLayer(clip_fc_new, yr_lyr)

    count_rasters = []

    for yr in years:
        arcpy.management.SelectLayerByAttribute(
            yr_lyr,
            "NEW_SELECTION",
            f"TRT_YEAR = {yr}"
        )

        tmp_ras = fr"memory\cnt_{yr}"
        if arcpy.Exists(tmp_ras):
            arcpy.management.Delete(tmp_ras)

        arcpy.conversion.PolygonToRaster(
            in_features=yr_lyr,
            value_field="MGMT_BIN",
            out_rasterdataset=tmp_ras,
            cell_assignment="CELL_CENTER",
            priority_field="NONE",
            cellsize=template_raster
        )

        count_rasters.append(Con(IsNull(Raster(tmp_ras)), 0, Raster(tmp_ras)))

    mgmt_times_treated = CellStatistics(count_rasters, "SUM", "DATA")
    mgmt_times_treated.save(mgmt_times_treated_path)

    # release references so ArcGIS can drop handles
    del mgmt_times_treated
    del count_rasters

    if arcpy.Exists(yr_lyr):
        arcpy.management.Delete(yr_lyr)

    clear_locks()

# =============================================================================
# SUMMARY
# =============================================================================

log("Done.")
log(f"Final vector output: {clip_fc_new}")

for _, out_path in out_rasters.items():
    if arcpy.Exists(out_path):
        log(f"Final raster: {out_path}")

if arcpy.Exists(mgmt_most_recent_year_path):
    log(f"Final raster: {mgmt_most_recent_year_path}")

if arcpy.Exists(mgmt_most_recent_time_since_path):
    log(f"Final raster: {mgmt_most_recent_time_since_path}")

if arcpy.Exists(mgmt_times_treated_path):
    log(f"Final raster: {mgmt_times_treated_path}")

for _, lut_path in lookup_tables.items():
    if arcpy.Exists(lut_path):
        log(f"Lookup table: {lut_path}")

RuntimeError: Output already exists. Delete or rename it first:
C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_vspy.gdb\ltdl_treatments_clip_new

In [1]:
import arcpy
import os

base_folder = r"C:\NCA_DATA\Ancillary_Data\LTDL"
vector_gdb = os.path.join(base_folder, "LTDL_vspy.gdb")

outputs = [
    os.path.join(vector_gdb, "ltdl_treatments_clip_new"),
    os.path.join(base_folder, "mgmt_binary_10m_new.tif"),
    os.path.join(base_folder, "mgmt_time_since_10m_new.tif"),
    os.path.join(base_folder, "mgmt_seeded_10m_new.tif"),
    os.path.join(base_folder, "mgmt_num_units_10m_new.tif"),
    os.path.join(base_folder, "mgmt_trt_major_10m_new.tif"),
    os.path.join(base_folder, "mgmt_trt_sub_10m_new.tif"),
    os.path.join(base_folder, "mgmt_treatment_type_10m_new.tif"),
    os.path.join(base_folder, "mgmt_most_recent_year_10m_new.tif"),
    os.path.join(base_folder, "mgmt_most_recent_time_since_10m_new.tif"),
    os.path.join(base_folder, "mgmt_times_treated_10m_new.tif"),
    os.path.join(vector_gdb, "lut_Trt_Type_Major_new"),
    os.path.join(vector_gdb, "lut_Trt_Type_Sub_new"),
    os.path.join(vector_gdb, "lut_Treatment_Type_new"),
]

for path in outputs:
    print(arcpy.Exists(path), path)

True C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_vspy.gdb\ltdl_treatments_clip_new
True C:\NCA_DATA\Ancillary_Data\LTDL\mgmt_binary_10m_new.tif
True C:\NCA_DATA\Ancillary_Data\LTDL\mgmt_time_since_10m_new.tif
True C:\NCA_DATA\Ancillary_Data\LTDL\mgmt_seeded_10m_new.tif
True C:\NCA_DATA\Ancillary_Data\LTDL\mgmt_num_units_10m_new.tif
True C:\NCA_DATA\Ancillary_Data\LTDL\mgmt_trt_major_10m_new.tif
True C:\NCA_DATA\Ancillary_Data\LTDL\mgmt_trt_sub_10m_new.tif
True C:\NCA_DATA\Ancillary_Data\LTDL\mgmt_treatment_type_10m_new.tif
True C:\NCA_DATA\Ancillary_Data\LTDL\mgmt_most_recent_year_10m_new.tif
True C:\NCA_DATA\Ancillary_Data\LTDL\mgmt_most_recent_time_since_10m_new.tif
True C:\NCA_DATA\Ancillary_Data\LTDL\mgmt_times_treated_10m_new.tif
True C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_vspy.gdb\lut_Trt_Type_Major_new
True C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_vspy.gdb\lut_Trt_Type_Sub_new
True C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_vspy.gdb\lut_Treatment_Type_new


This is an arcpy notebook with the ArcPro interpreter, so we can't really do much of the writing here if we want to get away from the geodatabase - proceed to LTDL_Analysis for writing NetCDF with rasters reclassified using the lookup tables

In [3]:
# =============================================================================
# Recover LTDL categorical lookup tables for already-written categorical rasters
# =============================================================================

import os
import csv
import json
from pathlib import Path

import arcpy


# =============================================================================
# USER INPUTS
# =============================================================================

input_gdb = r"C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_Release_Sept2024.gdb"
base_folder = r"C:\NCA_DATA\Ancillary_Data\LTDL"
vector_gdb_name = "LTDL_vspy.gdb"

aoi_fc = r"C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_assimilation\LTDL_assimilation.gdb\NCA_NAD83UTM11N"
template_raster = r"C:\NCA_DATA\templates\grid_template_10m_26911_uint8.tif"

treatment_polygons = os.path.join(input_gdb, "ltdl_treatment_polygons")
treatment_info = os.path.join(input_gdb, "treatment_info")

analysis_year = 2026


# =============================================================================
# OUTPUTS
# =============================================================================

lookup_folder = Path(base_folder) / "LTDL_lookup_tables"
lookup_folder.mkdir(parents=True, exist_ok=True)

# These are the categorical rasters already written by your existing workflow.
# The script does not rewrite them; it only creates lookup files.
lookup_specs = {
    "mgmt_trt_major_10m_new": "Trt_Type_Major",
    "mgmt_trt_sub_10m_new": "Trt_Type_Sub",
    "mgmt_treatment_type_10m_new": "Treatment_Type",
}


# =============================================================================
# HELPERS
# =============================================================================

def list_fields(table):
    return [f.name for f in arcpy.ListFields(table)]


def clean_label(value):
    if value is None:
        return None

    value = str(value).strip()

    if value in {"", " ", "None", "NULL", "<Null>", "nan", "NaN"}:
        return None

    return value


def find_field_case_insensitive(table, target_name):
    fields = list_fields(table)
    field_map = {f.lower(): f for f in fields}

    target_lower = target_name.lower()

    if target_lower not in field_map:
        raise ValueError(
            f"Could not find field '{target_name}' in:\n{table}\n\n"
            f"Available fields:\n{fields}"
        )

    return field_map[target_lower]


def build_sorted_lookup(table, label_field):
    """
    Builds code -> label lookup using sorted unique labels.

    This assumes your already-written categorical raster used integer codes
    derived from sorted unique categorical labels starting at 1.

    If your rasterization used an existing numeric code field, do not use this.
    """

    label_field = find_field_case_insensitive(table, label_field)

    labels = set()

    with arcpy.da.SearchCursor(table, [label_field]) as cursor:
        for (raw_label,) in cursor:
            label = clean_label(raw_label)

            if label is not None:
                labels.add(label)

    labels = sorted(labels)

    lookup = {
        code: label
        for code, label in enumerate(labels, start=1)
    }

    return lookup, label_field


def write_lookup_files(raster_stem, source_table, label_field, lookup):
    csv_path = lookup_folder / f"{raster_stem}_lookup.csv"
    json_path = lookup_folder / f"{raster_stem}_lookup.json"

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "raster_stem",
            "code",
            "label",
            "source_table",
            "source_field",
            "analysis_year"
        ])

        for code, label in lookup.items():
            writer.writerow([
                raster_stem,
                code,
                label,
                source_table,
                label_field,
                analysis_year
            ])

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "raster_stem": raster_stem,
                "source_table": source_table,
                "source_field": label_field,
                "analysis_year": analysis_year,
                "lookup": {str(code): label for code, label in lookup.items()}
            },
            f,
            indent=2
        )

    print(f"Wrote:")
    print(f"  {csv_path}")
    print(f"  {json_path}")


# =============================================================================
# RUN
# =============================================================================

print("\nChecking source tables...")
print(f"treatment_polygons exists: {arcpy.Exists(treatment_polygons)}")
print(f"treatment_info exists:     {arcpy.Exists(treatment_info)}")

if not arcpy.Exists(treatment_info):
    raise RuntimeError(f"Missing treatment_info table:\n{treatment_info}")

print("\nFields in treatment_info:")
for field in list_fields(treatment_info):
    print(f"  {field}")


for raster_stem, label_field in lookup_specs.items():

    print("\n" + "=" * 80)
    print(f"Recovering lookup for: {raster_stem}")
    print(f"Requested source field: {label_field}")

    lookup, resolved_field = build_sorted_lookup(
        table=treatment_info,
        label_field=label_field
    )

    print(f"Resolved field: {resolved_field}")
    print(f"Classes found: {len(lookup)}")

    for code, label in lookup.items():
        print(f"  {code}: {label}")

    write_lookup_files(
        raster_stem=raster_stem,
        source_table=treatment_info,
        label_field=resolved_field,
        lookup=lookup
    )

print("\nDone.")


Checking source tables...
treatment_polygons exists: True
treatment_info exists:     True

Fields in treatment_info:
  OBJECTID
  Prj_ID
  Trt_ID
  Plan_Imp
  Success
  Dates_Confirmed
  Init_Date
  Comp_Date
  Units
  Num_Units
  Trt_Type_Major
  Trt_Type_Sub
  Treatment_Type
  Treatment_Seeded
  Seed_List
  Seed_Mix_Name
  Seeds_or_Seedlings
  Seed_List_Confirmed
  Control_Areas
  Seed_Application_Rate
  Seed_Notes
  Objectives
  Planned_Implementation
  Actual_Implementation
  Treatment_Effect_and_Results
  Trt_Feature_Type
  Feature_Status
  How_Feature_Created
  Feature_Creation_Date
  Total_Acres
  BLM_Acres
  GIS_Notes
  Initiated_By
  Trt_Concerns
  Trt_Concerns_Desc
  Problems_Apply_Trt
  Problems_Apply_Trt_Desc
  Conditions_At_Trt
  Conditions_At_Trt_Desc
  Trt_Notes
  User_Updated

Recovering lookup for: mgmt_trt_major_10m_new
Requested source field: Trt_Type_Major
Resolved field: Trt_Type_Major
Classes found: 10
  1: Biological Control
  2: Closure/Exclosure
  3: Cultural 